In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold, LeaveOneOut, LeavePOut, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, make_scorer, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


iris = load_iris()
X = iris.data
y = iris.target

1. Изучите разбиение Leave-P-Out. Продемонстрируйте работу этого алгоритма на примере из лабораторной работы.

In [ ]:
lpo = LeavePOut(p=2)
print("Демонстрация Leave-P-Out (первые 5 итераций):")

for i, (train_index, test_index) in enumerate(lpo.split(X[:10])):
    if i >= 5:
        break
    print(f"Итерация {i+1}:")
    print(f"Train: index={train_index}")
    print(f"Test:  index={test_index}")

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

Демонстрация Leave-P-Out (первые 5 итераций):
Итерация 1:
Train: index=[2 3 4 5 6 7 8 9]
Test:  index=[0 1]
Итерация 2:
Train: index=[1 3 4 5 6 7 8 9]
Test:  index=[0 2]
Итерация 3:
Train: index=[1 2 4 5 6 7 8 9]
Test:  index=[0 3]
Итерация 4:
Train: index=[1 2 3 5 6 7 8 9]
Test:  index=[0 4]
Итерация 5:
Train: index=[1 2 3 4 6 7 8 9]
Test:  index=[0 5]


2. Изучите функцию cross_validate(). Продемонстрируйте работу этой функции на тех же данных.

In [ ]:
model = LogisticRegression(solver='liblinear', max_iter=200)

cv_results = cross_validate(
    estimator=model,
    X=X,
    y=y,
    cv=3,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True,
    return_estimator=True,
    n_jobs=-1
)

print("Результаты cross_validate:")
print("Test Accuracy: ", cv_results['test_accuracy'])
print("Test F1: ", cv_results['test_f1_macro'])
print("Train Accuracy: ", cv_results['train_accuracy'])
print("Train F1: ", cv_results['train_f1_macro'])
print("Среднее Accuracy: ", np.mean(cv_results['test_accuracy']))
print("Среднее F1: ", np.mean(cv_results['test_f1_macro']))
print("Время обучения (сек): ", cv_results['fit_time'])
print("Время предсказания (сек): ", cv_results['score_time'])

Результаты cross_validate:
Test Accuracy:  [0.96 0.96 0.94]
Test F1:  [0.95955882 0.95955882 0.94071491]
Train Accuracy:  [0.95 0.96 0.97]
Train F1:  [0.94984655 0.96019014 0.96963423]
Среднее Accuracy:  0.9533333333333333
Среднее F1:  0.9532775185052226
Время обучения (сек):  [0.0026052  0.00412369 0.00200725]
Время предсказания (сек):  [0.0047164  0.00892711 0.00414014]


3. Оцените при помощи кросс-валидации другие метрики эффективности для этой же модели.


In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro',
    'precision_macro': make_scorer(precision_score, average='macro'),
    'recall_macro': make_scorer(recall_score, average='macro')
}

cv_all_metrics = cross_validate(
    estimator=model,
    X=X,
    y=y,
    cv=3,
    scoring=scoring,
    n_jobs=-1
)

print("Оценка дополнительных метрик (усреднение macro):")
for metric in ['test_accuracy', 'test_f1_macro', 'test_precision_macro', 'test_recall_macro']:
    values = cv_all_metrics[metric]
    print(f"{metric}: {values} (среднее: {np.mean(values):.3f})")

Оценка дополнительных метрик (усреднение macro):
test_accuracy: [0.96 0.96 0.94] (среднее: 0.953)
test_f1_macro: [0.95955882 0.95955882 0.94071491] (среднее: 0.953)
test_precision_macro: [0.96296296 0.95955882 0.95      ] (среднее: 0.958)
test_recall_macro: [0.96078431 0.95955882 0.94117647] (среднее: 0.954)


4. Сравните кросс-валидированные результаты работы нескольких моделей на одних и тех же данных.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=200),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

print("Сравнение моделей на датасете Iris:")
for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy', n_jobs=-1)
    print(f"{name}:")
    print(f"  Accuracy по фолдам: {cv_scores}")
    print(f"  Средняя Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

Сравнение моделей на датасете Iris:
Logistic Regression:
  Accuracy по фолдам: [1.         0.93333333 0.93333333 0.96666667 0.96666667]
  Средняя Accuracy: 0.9600 (+/- 0.0249)
Decision Tree:
  Accuracy по фолдам: [1.         0.96666667 0.93333333 0.93333333 0.93333333]
  Средняя Accuracy: 0.9533 (+/- 0.0267)
SVM:
  Accuracy по фолдам: [1.         1.         0.93333333 0.93333333 0.96666667]
  Средняя Accuracy: 0.9667 (+/- 0.0298)
KNN:
  Accuracy по фолдам: [1.         1.         0.96666667 0.93333333 0.96666667]
  Средняя Accuracy: 0.9733 (+/- 0.0249)


5. Повторите анализ на другом датасете: встроенном наборе данных о диабете.

In [ ]:
diabetes = load_diabetes()
X_diabetes, y_diabetes = diabetes.data, diabetes.target

y_diabetes_binary = (y_diabetes > np.median(y_diabetes)).astype(int)

print("Датасет о диабете:")
print(f"Размер данных: {X_diabetes.shape}")
print(f"Количество классов: {len(np.unique(y_diabetes_binary))}")
print(f"Распределение классов: 0 - {sum(y_diabetes_binary==0)}, 1 - {sum(y_diabetes_binary==1)}")

print("\nСравнение моделей на датасете о диабете:")
for name, model in models.items():
    cv_scores = cross_val_score(model, X_diabetes, y_diabetes_binary, cv=kf, scoring='accuracy', n_jobs=-1)
    print(f"{name}:")
    print(f"  Accuracy по фолдам: {cv_scores}")
    print(f"  Средняя Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

scoring = {
    'accuracy': 'accuracy',
    'f1': 'f1',
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score)
}

print("\nОценка нескольких метрик для Logistic Regression на датасете о диабете:")
log_reg_model = LogisticRegression(solver='liblinear', max_iter=200)

cv_results_diabetes = cross_validate(
    estimator=log_reg_model,
    X=X_diabetes,
    y=y_diabetes_binary,
    cv=kf,
    scoring=scoring,
    n_jobs=-1
)

for metric in ['test_accuracy', 'test_f1', 'test_precision', 'test_recall']:
    values = cv_results_diabetes[metric]
    print(f"{metric}: {values} (среднее: {np.mean(values):.4f})")

Датасет о диабете:
Размер данных: (442, 10)
Количество классов: 2
Распределение классов: 0 - 221, 1 - 221

Сравнение моделей на датасете о диабете:
Logistic Regression:
  Accuracy по фолдам: [0.74157303 0.83146067 0.67045455 0.73863636 0.70454545]
  Средняя Accuracy: 0.7373 (+/- 0.0537)
Decision Tree:
  Accuracy по фолдам: [0.6741573  0.74157303 0.54545455 0.625      0.67045455]
  Средняя Accuracy: 0.6513 (+/- 0.0647)
SVM:
  Accuracy по фолдам: [0.75280899 0.80898876 0.68181818 0.76136364 0.71590909]
  Средняя Accuracy: 0.7442 (+/- 0.0430)
KNN:
  Accuracy по фолдам: [0.68539326 0.71910112 0.69318182 0.65909091 0.70454545]
  Средняя Accuracy: 0.6923 (+/- 0.0201)

Оценка нескольких метрик для Logistic Regression на датасете о диабете:
test_accuracy: [0.74157303 0.83146067 0.67045455 0.73863636 0.70454545] (среднее: 0.7373)
test_f1: [0.71604938 0.84210526 0.70103093 0.73563218 0.68292683] (среднее: 0.7355)
test_precision: [0.70731707 0.83333333 0.73913043 0.72727273 0.66666667] (среднее: 

6. Сделайте k-блочную перекрёстную проверку (KFold) модели логистической регрессии, предварительно стандартизировав данные. Для этого создайте конвейер с помощью make_pipeline из библиотеки sklearn.pipeline, который будет стандартизировать, а затем выполнять логистическую регрессию.

In [ ]:
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='liblinear', max_iter=200)
)

print("Конвейер создан: StandardScaler -> LogisticRegression")

print("\nKFold со стандартизацией на датасете о диабете:")
cv_scores_pipeline = cross_val_score(pipeline, X_diabetes, y_diabetes_binary, cv=kf, scoring='accuracy', n_jobs=-1)
print(f"Accuracy по фолдам: {cv_scores_pipeline}")
print(f"Средняя Accuracy: {np.mean(cv_scores_pipeline):.4f} (+/- {np.std(cv_scores_pipeline):.4f})")

print("\nСравнение с моделью без стандартизации:")
model_no_scale = LogisticRegression(solver='liblinear', max_iter=200)
cv_scores_no_scale = cross_val_score(model_no_scale, X_diabetes, y_diabetes_binary, cv=kf, scoring='accuracy', n_jobs=-1)
print(f"Без стандартизации: {np.mean(cv_scores_no_scale):.4f} (+/- {np.std(cv_scores_no_scale):.4f})")
print(f"Со стандартизацией: {np.mean(cv_scores_pipeline):.4f} (+/- {np.std(cv_scores_pipeline):.4f})")

print("\nОценка нескольких метрик с конвейером на датасете о диабете:")
cv_results_pipeline = cross_validate(
    estimator=pipeline,
    X=X_diabetes,
    y=y_diabetes_binary,
    cv=kf,
    scoring=scoring,
    n_jobs=-1
)

for metric in ['test_accuracy', 'test_f1', 'test_precision', 'test_recall']:
    values = cv_results_pipeline[metric]
    print(f"{metric}: {values} (среднее: {np.mean(values):.4f})")

Конвейер создан: StandardScaler -> LogisticRegression

KFold со стандартизацией на датасете о диабете:
Accuracy по фолдам: [0.73033708 0.85393258 0.68181818 0.70454545 0.71590909]
Средняя Accuracy: 0.7373 (+/- 0.0604)

Сравнение с моделью без стандартизации:
Без стандартизации: 0.7373 (+/- 0.0537)
Со стандартизацией: 0.7373 (+/- 0.0604)

Оценка нескольких метрик с конвейером на датасете о диабете:
test_accuracy: [0.73033708 0.85393258 0.68181818 0.70454545 0.71590909] (среднее: 0.7373)
test_f1: [0.70731707 0.86315789 0.7254902  0.69047619 0.69135802] (среднее: 0.7356)
test_precision: [0.69047619 0.85416667 0.7254902  0.70731707 0.68292683] (среднее: 0.7321)
test_recall: [0.725      0.87234043 0.7254902  0.6744186  0.7       ] (среднее: 0.7394)
